#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

In [2]:
WEIGHTS_DIR = "../weights"

In [3]:
weights = ConvNeXt_Small_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification\train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification\val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Train classes: {class_names}")
print(f"Validation classes: {val_dataset.classes}")
print(f"Number of classes: {num_classes}")



Train classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']
Validation classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']
Number of classes: 4


In [5]:
model = convnext_small(weights=weights)


In [6]:
in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(in_features, num_classes)


#### Training 1 (Freeze-Backbone)

In [7]:
for param in model.features.parameters():
    param.requires_grad = False


In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-4
)


In [10]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss / len(loader), train_acc


In [11]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss / len(loader), val_acc

In [12]:
EPOCHS_STAGE_FREEZE = 8

print("\n-----------Starting Phase 1 Training-----------\n")


for epoch in range(EPOCHS_STAGE_FREEZE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")



-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [00:31<00:00,  3.88s/it]



Epoch [1/8]
Train Loss: 1.3767 | Train Acc: 0.3292
Val Loss: 1.3219 | Val Acc: 0.3875
Precision: 0.4750 | Recall: 0.3875 | F1: 0.3178




Validating: 100%|██████████| 8/8 [00:30<00:00,  3.79s/it]



Epoch [2/8]
Train Loss: 1.2348 | Train Acc: 0.5062
Val Loss: 1.1817 | Val Acc: 0.5458
Precision: 0.5234 | Recall: 0.5458 | F1: 0.4850




...


Epoch [6/8]
Train Loss: 0.8576 | Train Acc: 0.7760
Val Loss: 0.8330 | Val Acc: 0.6833
Precision: 0.8140 | Recall: 0.6833 | F1: 0.6364




Validating: 100%|██████████| 8/8 [00:29<00:00,  3.74s/it]



Epoch [7/8]
Train Loss: 0.8037 | Train Acc: 0.8031
Val Loss: 0.7825 | Val Acc: 0.7000
Precision: 0.7798 | Recall: 0.7000 | F1: 0.6626




Validating: 100%|██████████| 8/8 [00:30<00:00,  3.84s/it]


Epoch [8/8]
Train Loss: 0.7557 | Train Acc: 0.8156
Val Loss: 0.7340 | Val Acc: 0.7292
Precision: 0.7970 | Recall: 0.7292 | F1: 0.7034




#### Training 2 (Fine-Tuning)

In [13]:
for param in model.parameters():
    param.requires_grad = True


In [14]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

In [15]:
EPOCHS_STAGE_FINETUNE = 20

best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_STAGE_FINETUNE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

        
torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ConvNeXt-S.pth"))


-----------Starting Fine-tuning Stage-----------



Validating: 100%|██████████| 8/8 [00:34<00:00,  4.36s/it]



Epoch [1/20]
Train Loss: 0.5389 | Train Acc: 0.8552
Val Loss: 0.4380 | Val Acc: 0.8417
Precision: 0.8505 | Recall: 0.8417 | F1: 0.8393


Validating: 100%|██████████| 8/8 [00:34<00:00,  4.36s/it]



Epoch [2/20]
Train Loss: 0.3380 | Train Acc: 0.9156
Val Loss: 0.3205 | Val Acc: 0.8792
Precision: 0.8812 | Recall: 0.8792 | F1: 0.8789


...


Epoch [18/20]
Train Loss: 0.0303 | Train Acc: 0.9917
Val Loss: 0.0877 | Val Acc: 0.9792
Precision: 0.9808 | Recall: 0.9792 | F1: 0.9794


Validating: 100%|██████████| 8/8 [00:34<00:00,  4.32s/it]



Epoch [19/20]
Train Loss: 0.0257 | Train Acc: 0.9958
Val Loss: 0.0903 | Val Acc: 0.9833
Precision: 0.9844 | Recall: 0.9833 | F1: 0.9834


Validating: 100%|██████████| 8/8 [00:34<00:00,  4.30s/it]



Epoch [20/20]
Train Loss: 0.0221 | Train Acc: 0.9958
Val Loss: 0.0911 | Val Acc: 0.9792
Precision: 0.9808 | Recall: 0.9792 | F1: 0.9794


#### Prediction Sample

In [16]:
from PIL import Image

In [17]:
checkpoint = torch.load("../weights/ConvNeXt-S.pth")

class_names = checkpoint["class_names"]

model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(image_path):

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


C:\Users\yugil\AppData\Local\Temp\ipykernel_25232\281767184.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../weights/ConvNeXt-S.pth")


In [18]:
# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\johnp\\Desktop\\Dataset\\Cassava_Leaf_Datasets\\Classification\\val\\Healthy\\Healthy_val_13.jpg'